# Lab Ngày 26: Hạ Tầng MCP/A2A & Agentic Routing

**Khóa học:** AICB-P2T2 · Tuần 6 · Chương 6  
**Framework:** [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/)  
**Thời lượng:** ~2 giờ

---

## Câu hỏi mở đầu

> *"Agent gọi agent — hạ tầng của bạn có scale được cho multi-agent choreography không?"*

Một hệ thống nghiên cứu cần 4 agent chuyên biệt phối hợp. Nếu không có chuẩn giao tiếp chung, mỗi agent nói "ngôn ngữ" riêng. **MCP + A2A** là giao thức phổ quát cho hệ sinh thái AI agent.

---

## Mục tiêu học tập

Sau buổi lab này, bạn sẽ:

1. Thiết kế và triển khai **hạ tầng MCP server** cho AI tools
2. Triển khai **giao tiếp A2A** giữa nhiều agent bằng ADK
3. Xây dựng **lớp agentic routing** với semantic routing
4. Áp dụng các mẫu **state, bảo mật và observability** trong hệ multi-agent

## Sản phẩm cuối ngày

| Thành phần | Yêu cầu |
|------------|--------|
| MCP server | 3 tools qua stdio (local) hoặc SSE (remote) |
| Agent registry | Health check + khám phá capability |
| Semantic router | Định tuyến request tới đúng specialist |
| Demo multi-agent | Orchestrator + 2 specialist qua A2A |
| Trace | Toàn bộ luồng multi-agent hiển thị trong log/trace |
| Data governance | Policy MCP/A2A + audit log + HITL gate |

---
## Module 0 — Cài đặt môi trường

Lab dùng **Conda** (môi trường khuyến nghị: `pii-env`). Trước khi chạy cell bên dưới:

```bash
conda create -n pii-env python=3.12 -y   # chỉ lần đầu
conda activate pii-env
```

Chọn kernel **pii-env** trong Jupyter nếu notebook hỏi.

Cài ADK kèm **extra A2A** và MCP SDK. Gói `google-adk` cơ bản **không** bao gồm dependency A2A.

In [1]:
# Chạy một lần sau khi conda activate pii-env (kernel Jupyter = pii-env)
!pip install -q -r requirements.txt

# Nếu vẫn thấy cảnh báo cryptography từ presidio cũ (2.2.360), nâng cấp:
# !pip install -q -U "presidio-anonymizer>=2.2.363"

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

# Google AI Studio (khuyến nghị cho lab)
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")

assert os.getenv("GOOGLE_API_KEY"), (
    "Đặt GOOGLE_API_KEY trong .env — lấy key tại https://aistudio.google.com/app/apikey"
)

print("✓ Môi trường sẵn sàng")
print(f"  Thư mục dự án: {PROJECT_ROOT}")

✓ Môi trường sẵn sàng
  Thư mục dự án: /Users/harrietlesly/Day26-MCP_A2A_Infrastructure


### Tổng quan kiến trúc

```
┌─────────────────────────────────────────────────────────────┐
│                     ORCHESTRATOR (ADK)                       │
│  Semantic Router → ủy quyền cho các specialist               │
└──────────┬──────────────────────────────┬───────────────────┘
           │ A2A (HTTP)                   │ MCP (stdio/SSE)
           ▼                              ▼
┌──────────────────┐            ┌──────────────────────┐
│  Search Agent    │            │  MCP Tools Server     │
│  :8001           │            │  search / sql / sum   │
├──────────────────┤            └──────────────────────┘
│  Database Agent  │
│  :8002           │
└──────────────────┘
```

**Điểm then chốt:** MCP chuẩn hóa truy cập *tool*; A2A chuẩn hóa giao tiếp *agent*. ADK kết nối cả hai.

---
## Buoc bat buoc - Khoi dong A2A Specialist Servers

**Phai lam truoc Module 2, Module 4 va Capstone.** Neu bo qua, kiem tra agent card se bao `[CHUA DAT]`.

**Dieu kien:** da cai dependency (Module 0) va dang o thu muc du an. Kich hoat conda env:

```bash
conda activate pii-env
export PYTHONPATH=$PWD
```

### Cach 1 - Tu notebook (khuyen nghi)

Chay cell ben duoi de kiem tra/khoi dong ca ba server nen. Doi thong bao `search_agent`, `database_agent`, `synthesis_agent` deu san sang, roi chay cell kiem tra.

### Cach 2 - Ba terminal rieng

```bash
cd Day26-MCP_A2A_Infrastructure
conda activate pii-env
export PYTHONPATH=$PWD

# Terminal 1
python -m uvicorn agents.search_agent.agent:a2a_app --host localhost --port 8001

# Terminal 2
python -m uvicorn agents.database_agent.agent:a2a_app --host localhost --port 8002

# Terminal 3
python -m uvicorn agents.synthesis_agent.agent:a2a_app --host localhost --port 8003
```

Hoac: `bash scripts/start_search_agent.sh`, `bash scripts/start_database_agent.sh`, `bash scripts/start_synthesis_agent.sh`.

### Cach 3 - Mot lenh nen

```bash
bash scripts/start_a2a_servers.sh
```

### Kiem tra / Dung

```bash
curl http://localhost:8001/.well-known/agent-card.json
curl http://localhost:8002/.well-known/agent-card.json
curl http://localhost:8003/.well-known/agent-card.json
bash scripts/stop_a2a_servers.sh
```


In [ ]:
# Khoi dong A2A servers nen (chay cell nay mot lan moi phien lab)
import subprocess
import time
from pathlib import Path

import httpx

STARTUP_CARDS = {
    "search_agent": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8003/.well-known/agent-card.json",
}

def _check_cards(timeout=2):
    errors = []
    for name, url in STARTUP_CARDS.items():
        try:
            r = httpx.get(url, timeout=timeout)
            r.raise_for_status()
        except Exception as exc:
            errors.append(f"{name}: {exc}")
    return errors

errors = _check_cards()
if not errors:
    print("OK - A2A servers already running: search_agent, database_agent, synthesis_agent")
else:
    script = PROJECT_ROOT / "scripts" / "start_a2a_servers.sh"
    if script.exists():
        try:
            result = subprocess.run(
                ["bash", str(script)],
                cwd=PROJECT_ROOT,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=90,
            )
            print(result.stdout)
            if result.stderr:
                print(result.stderr)
        except Exception as exc:
            print(f"Could not run startup script automatically: {exc}")
            print("Run the 3 uvicorn commands manually from the instructions above.")
    else:
        print("Startup script not found. Run the 3 uvicorn commands manually from the instructions above.")

    for _ in range(10):
        errors = _check_cards()
        if not errors:
            break
        time.sleep(1)

    if errors:
        print("CHUA DAT - A2A servers are not ready:")
        for err in errors:
            print("  -", err)
    else:
        print("OK - A2A servers ready")


In [ ]:
# Kiểm tra agent card — phải thấy ✓ trước khi sang Module 4 / Capstone
import httpx

AGENT_CARDS = {
    "search_agent": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8003/.well-known/agent-card.json",
}

all_ok = True
for name, url in AGENT_CARDS.items():
    try:
        r = httpx.get(url, timeout=5)
        r.raise_for_status()
        card = r.json()
        print(f"✓ {name}: {card.get('name')} — {str(card.get('description', ''))[:50]}...")
    except Exception as exc:
        all_ok = False
        print(f"✗ {name}: {exc}")
        port = {"search_agent": 8001, "database_agent": 8002, "synthesis_agent": 8003}[name]
        print(f"  → Chạy: uvicorn agents.{name}.agent:a2a_app --host localhost --port {port}")

if all_ok:
    print("\n✓ A2A servers sẵn sàng — tiếp tục lab")
else:
    print("\n✗ Chưa sẵn sàng — chạy: bash scripts/start_a2a_servers.sh")

---
## Module 1 — Model Context Protocol (MCP)

### Lý thuyết (từ slide)

| Lớp | Vai trò |
|-----|--------|
| **Tools Server** | Thực thi hàm (search, SQL, API) |
| **Resources Server** | Expose dữ liệu (file, hàng DB) |
| **Prompts Server** | Template prompt tái sử dụng |

**Transport:** `stdio` (dev local), `HTTP+SSE` (remote/K8s), `WebSocket` (streaming hai chiều)

**Quy tắc thiết kế:**
- Docstring rõ ràng (LLM đọc để chọn tool)
- Type hint cho mọi tham số
- Validate input trước khi thực thi
- SQL tool mặc định chỉ đọc (read-only)
- Log mọi lần gọi tool

### 📝 Bài tập 1.1 — Khám phá MCP Server

Mở `mcp_server/research_tools_server.py` và trả lời:
1. Ba tool nào được expose?
2. `_sql_query` enforce governance như thế nào?
3. Vì sao dùng transport stdio khi dev local?

In [ ]:
# Demo: call MCP tool logic directly (no server startup needed)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "mcp_server"))

from research_tools_server import _count_words, _search_documents, _sql_query, _summarize_text

print("=== search_documents ===")
print(_search_documents("MCP"))

print("
=== sql_query (allowed) ===")
print(_sql_query("SELECT * FROM agent_metrics"))

print("
=== sql_query (blocked) ===")
try:
    _sql_query("DROP TABLE agent_metrics")
except ValueError as exc:
    print(f"Blocked: {exc}")

print("
=== summarize_text ===")
print("
".join(_summarize_text("MCP build once. A2A connects agents. Routing chooses specialist.")))

print("
=== count_words ===")
print(_count_words("MCP build once. A2A connects agents."))


### Ket noi MCP Tools voi ADK Agent

ADK wraps the MCP server with `McpToolset`. The repo orchestrator binds 4 tools: `search_documents`, `sql_query`, `summarize_text`, `count_words`.


In [ ]:
import sys

from google.adk.tools.mcp_tool import StdioConnectionParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from mcp import StdioServerParameters

mcp_server_path = PROJECT_ROOT / "mcp_server" / "research_tools_server.py"

mcp_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,
            args=[str(mcp_server_path)],
        ),
        timeout=10,
    ),
    tool_filter=["search_documents", "sql_query", "summarize_text", "count_words"],
)

print("OK - McpToolset configured")
print(f"  Server: {mcp_server_path.name}")
print("  Filter: search_documents, sql_query, summarize_text, count_words")


### Bai tap 1.2 - Them tool MCP thu tu

**Nhiem vu:** Mo rong `research_tools_server.py` voi tool `count_words` tra ve so tu trong chuoi.

Checklist:
- [x] Them vao `list_tools()` voi `inputSchema` dung
- [x] Trien khai trong `call_tool()`
- [x] Them vao `policy.json` de orchestrator nap `tool_filter` dong
- [x] Test qua ADK agent hoac goi ham truc tiep


In [ ]:
# Kiem tra Bai tap 1.2 - tool count_words
import asyncio

from research_tools_server import call_tool, list_tools

async def _test_count_words():
    tools = await list_tools()
    print("Tools:", [t.name for t in tools])
    result = await call_tool("count_words", {"text": "MCP build mot lan A2A ket noi agent"})
    print("count_words ->", result[0].text)

await _test_count_words()


---
## Module 2 — Giao thức Agent-to-Agent (A2A)

### Lý thuyết

A2A coi agent như **microservices**:

| Khái niệm | Mô tả |
|-----------|-------|
| **Agent Card** | JSON tại `/.well-known/agent-card.json` — capability & schema |
| **Task** | Đơn vị công việc với các trạng thái vòng đời |
| **Message** | Giao tiếp trong một task |

### Vòng đời Task

```
Submitted → Working → Input Required → Completed
                  ↘ Failed / Canceled
```

Task có thể **tạm dừng** ở `input-required` — agent yêu cầu caller cung cấp thêm thông tin trước khi tiếp tục.

### Ánh xạ sang ADK

| Hành động | API ADK |
|----------|--------|
| Expose agent làm A2A server | `to_a2a(agent, port=8001)` |
| Tiêu thụ agent remote | `RemoteA2aAgent(name, description, agent_card=url)` |
| Thay thế qua CLI | `adk api_server --a2a --port 8001 agents/search_agent` |

### 🚀 Thực hành — Kiểm tra lại A2A (Module 2)

> **Đã khởi động ở Module 0.5 chưa?** Nếu chưa, quay lại phầ **「Bước bắt buộc — Khởi động A2A Specialist Servers」** và chạy cell khởi động + kiểm tra.

Cell bên dưới lặp lại bước xác nhận agent card (giống Module 0.5).

In [ ]:
import httpx

AGENT_CARDS = {
    "search_agent": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8003/.well-known/agent-card.json",
}

for name, url in AGENT_CARDS.items():
    try:
        response = httpx.get(url, timeout=3)
        response.raise_for_status()
        card = response.json()
        print(f"OK - {name}: {card.get('name')} - {card.get('description', '')[:60]}...")
    except Exception as exc:
        port = {"search_agent": 8001, "database_agent": 8002, "synthesis_agent": 8003}[name]
        print(f"FAIL - Cannot connect to {name}: {exc}")
        print(f"  Start: uvicorn agents.{name}.agent:a2a_app --port {port}")


### Tiêu thụ Remote Agent từ Orchestrator

Orchestrator dùng `RemoteA2aAgent` làm sub-agent. LLM coordinator của ADK ủy quyền dựa trên `description` của từng sub-agent.

In [ ]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

remote_search = RemoteA2aAgent(
    name="search_agent",
    description="Search documents and web evidence for research tasks.",
    agent_card=AGENT_CARDS["search_agent"],
)

remote_database = RemoteA2aAgent(
    name="database_agent",
    description="Run read-only SQL over agent metrics.",
    agent_card=AGENT_CARDS["database_agent"],
)

remote_synthesis = RemoteA2aAgent(
    name="synthesis_agent",
    description="Synthesize findings into a structured final report.",
    agent_card=AGENT_CARDS["synthesis_agent"],
)

print("OK - RemoteA2aAgent proxies configured")
for name, url in AGENT_CARDS.items():
    print(f"  {name} -> {url}")


### 📝 Bài tập 2.1 — A2A vs Sub-Agent Local

Điền bảng theo hiểu biết của bạn:

| Tiêu chí | A2A (Remote) | Sub-Agent Local |
|----------|-------------|------------------|
| Triển khai | Chạy như service HTTP độc lập (`uvicorn`, port riêng vd :8001-:8003), publish agent card `.well-known/agent-card.json`, cần khởi động + quản lý lifecycle riêng (`start_a2a_servers.sh`) | Khai báo trực tiếp trong `sub_agents=[local_agent]` của agent cha, chạy chung process/session, không cần server hay port riêng |
| Hiệu năng | Có overhead mạng: HTTP round-trip + serialize/deserialize theo A2A protocol (JSON-RPC-like) — độ trễ cao hơn, phụ thuộc network | Gọi trực tiếp trong cùng process (function call/coroutine) — gần như không overhead mạng, độ trễ thấp hơn đáng kể |
| Cô lập state | State/session của mỗi agent hoàn toàn tách biệt qua ranh giới network — crash hoặc lỗi 1 agent không kéo sập agent khác, phù hợp multi-team/multi-tenant | State chia sẻ trong cùng `ToolContext`/session của agent cha — tiện lợi truyền context nhưng dễ rò rỉ state chéo giữa các sub-agent, ít cô lập hơn |
| Phù hợp khi | Specialist do team/repo khác triển khai, cần scale ngang độc lập theo tải riêng (vd `search_agent` tải cao hơn `database_agent`), cần governance/audit theo ranh giới network rõ ràng, hoặc viết bằng stack khác | Prototype nhanh, agent đơn giản trong cùng codebase/team, ưu tiên độ trễ thấp và triển khai tối giản (không cần vận hành nhiều service) |

**Thảo luận:** Khi nào chọn A2A thay vì `sub_agents=[local_agent]` trong ADK?

Chọn **A2A** khi cần: (1) cô lập triển khai — mỗi specialist versioning/deploy độc lập, không phụ thuộc chung một release của orchestrator; (2) scale ngang độc lập theo agent (vd nhiều instance `search_agent` phía sau load balancer trong khi `database_agent` chỉ cần 1 instance); (3) ranh giới governance rõ ràng theo network boundary — dễ audit request/response qua HTTP, dễ enforce policy như trong lab này (`authorize_a2a_dispatch`, `require_trace_id`); (4) team khác nhau sở hữu từng specialist, có thể dùng framework/ngôn ngữ khác ADK.

Chọn **sub-agent local** khi hệ thống còn nhỏ, tất cả agent nằm trong cùng team/codebase, độ trễ thấp là ưu tiên hàng đầu, và không cần vận hành thêm hạ tầng (port, health check, service discovery) cho từng specialist.


---
## Module 3 — Mẫu Agentic Routing

### So sánh chiến lược routing

| Chiến lược | Ưu điểm | Nhược điểm | Khi nào dùng |
|------------|---------|------------|-------------|
| **Keyword** | Nhanh, đơn giản | Dễ vỡ, bỏ sót ngữ cảnh | ≤5 agent |
| **Embedding** | Bền, scale tốt | Chậm hơn, cần embedding | 5–50 agent |
| **LLM-based** | Linh hoạt, hiểu ngữ cảnh | Đắt, chậm | Routing phức tạp |

**Semantic routing:** embed request → cosine similarity với mô tả capability của agent → top-k ứng viên.

**Fallback chain:** agent chính → agent dự phòng → leo thang lên người — không để user bị kẹt.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.semantic_router import AgentCapability, SemanticRouter

router = SemanticRouter(
    agents=[
        AgentCapability(
            name="search_agent",
            description="Search web articles documents MCP A2A evidence",
            tags=["search", "web", "documents", "tim", "bai_viet", "mcp", "a2a"],
        ),
        AgentCapability(
            name="database_agent",
            description="SQL metrics database SELECT agent_metrics latency average",
            tags=["sql", "metrics", "database", "select", "agent_metrics", "latency", "do_tre", "avg"],
        ),
        AgentCapability(
            name="synthesis_agent",
            description="Summarize synthesize findings final executive report",
            tags=["summary", "report", "synthesis", "tom_tat", "tong_hop", "bao_cao", "executive"],
        ),
    ]
)

test_queries = [
    "Tim bai viet ve viec ap dung giao thuc MCP",
    "SELECT do tre trung binh tu agent_metrics",
    "Viet tom tat mot trang ve ket qua nghien cuu",
    "Xin chao, ban lam duoc gi?",
]

print(f"{'Request':<50} {'Route target':<20} Score")
print("-" * 80)
for query in test_queries:
    candidates = router.route(query, top_k=1)
    target = router.route_with_fallback(query, fallback="orchestrator")
    score = candidates[0][1] if candidates else 0.0
    print(f"{query[:48]:<50} {target:<20} {score:.3f}")


### Mẫu Agent Registry

Trên production, agent đăng ký khi khởi động và hủy đăng ký khi tắt. Registry lưu metadata capability để discovery.

In [ ]:
from lab_utils.agent_registry import AgentRegistry, RegisteredAgent

registry = AgentRegistry()

registry.register(
    RegisteredAgent(
        name="search_agent",
        url="http://localhost:8001",
        description="Web and document search specialist",
        capabilities={"skills": ["search_web"], "transport": "a2a"},
    )
)
registry.register(
    RegisteredAgent(
        name="database_agent",
        url="http://localhost:8002",
        description="Read-only SQL metrics specialist",
        capabilities={"skills": ["run_sql_query"], "transport": "a2a"},
    )
)
registry.register(
    RegisteredAgent(
        name="synthesis_agent",
        url="http://localhost:8003",
        description="Report synthesis specialist",
        capabilities={"skills": ["synthesize_report"], "transport": "a2a"},
    )
)

print("Registered agents:")
for agent in registry.list_agents():
    status = "healthy" if agent.healthy else "unhealthy"
    print(f"  - {agent.name} ({status}) @ {agent.url}")

print("
Find capability 'sql':")
for agent in registry.find_by_capability("sql"):
    print(f"  -> {agent.name}")


### 📝 Bài tập 3.1 — Xây dựng Fallback Chain

**Nhiệm vụ:** Mở rộng `SemanticRouter.route_with_fallback` để nhận danh sách fallback có thứ tự:

```python
def route_with_chain(self, request: str, chain: list[str]) -> str:
    """Thử route chính; nếu điểm < ngưỡng, đi theo chuỗi fallback."""
    ...
```

Test với: `chain=["search_agent", "database_agent", "orchestrator"]`

In [ ]:
# Kiem tra Bai tap 3.1 - route_with_chain
from lab_utils.routing_tool import _router as router

chain = ["search_agent", "database_agent", "orchestrator"]

test_requests = [
    "Tim bai viet ve MCP tren web",
    "SELECT avg latency FROM agent_metrics",
    "abcxyz khong lien quan gi ca",
]
for req in test_requests:
    top = router.route(req, top_k=1)
    result = router.route_with_chain(req, chain)
    print(f"{req!r:50} top={top} -> route_with_chain={result}")


---
## Module 4 - Full Flow Demo

### Orchestrator Pattern

```
User request
     |
     v
Orchestrator (decompose + route)
     |---> Search Agent (A2A :8001)
     |---> Database Agent (A2A :8002)
     |---> Synthesis Agent (A2A :8003)
     `---> MCP Tools (stdio: search / sql / summarize / count_words)
```

**Requirement:** Module 0.5 is running - A2A servers OK on ports 8001/8002/8003.

The examples below run **orchestrator -> A2A + MCP -> synthesis** through `run_full_flow()`.


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.full_flow import check_a2a_servers, print_flow_summary, run_full_flow

ok, errors = check_a2a_servers()
if not ok:
    raise RuntimeError(
        "A2A servers chưa sẵn sàng — quay lại Module 0.5.\n" + "\n".join(errors)
    )
print("✓ A2A servers OK — sẵn sàng chạy full luồng")

### Ví dụ 1 — A2A: Orchestrator → Search Agent

Luồng: user → orchestrator → **transfer A2A** → `search_agent` → tổng hợp.

In [ ]:
# Ví dụ 1 — A2A search delegation
result_1 = await run_full_flow(
    "Transfer sang search_agent để tìm kiếm web về multi-agent orchestration.",
    verbose=True,
)
print_flow_summary(result_1)

### Ví dụ 2 — MCP + A2A: Tài liệu + Metrics + Tổng hợp

Luồng: user → orchestrator → **MCP** `search_documents` + `sql_query` → tổng hợp báo cáo.

*(Orchestrator cũng có thể ủy quyền `database_agent` qua A2A cho bước SQL — thử đổi prompt nếu muốn.)*

In [ ]:
# Ví dụ 2 — MCP multi-tool + tổng hợp
result_2 = await run_full_flow(
    "Bước 1: dùng search_documents tìm MCP. "
    "Bước 2: dùng sql_query SELECT * FROM agent_metrics. "
    "Bước 3: tóm tắt kết quả thành báo cáo ngắn.",
    verbose=True,
)
print_flow_summary(result_2)

### Capstone Demo — ADK Web UI (sinh viên tự chạy)

ADK Web là giao diện chat tương tác cho orchestrator. Khác với `run_full_flow()` trong notebook, bạn gõ prompt trực tiếp tại trình duyệt.

#### Kiến trúc khi chạy capstone

```
localhost:8000  ADK Web (orchestrator)
     ├── A2A → search_agent      :8001
     ├── A2A → database_agent    :8002
     ├── A2A → synthesis_agent   :8003
     └── MCP   research_tools    (stdio, spawn tự động)
```

#### Cách khởi động (khuyến nghị — một lệnh)

```bash
cd Day26-MCP_A2A_Infrastructure
conda activate pii-env
export PYTHONPATH=$PWD
bash scripts/start_capstone.sh
```

Script trên sẽ: (1) khởi động 3 A2A specialist nền, (2) mở ADK Web tại http://localhost:8000.

#### Cách khởi động thủ công (2 terminal)

**Terminal 1 — A2A specialists:**
```bash
bash scripts/start_a2a_servers.sh
```

**Terminal 2 — ADK Web:**
```bash
bash scripts/start_adk_web.sh
# hoặc: adk web agents/orchestrator
```

```bash
# ❌ SAI — sẽ báo "No root_agent found for 'agents'"
adk web agents
```

#### Công cụ mới trong capstone

| Thành phần | File | Vai trò |
|------------|------|--------|
| `start_capstone.sh` | `scripts/` | Khởi động A2A + ADK Web một lệnh |
| `synthesis_agent` | `agents/synthesis_agent/` | Specialist tổng hợp báo cáo (:8003) |
| `suggest_routing` | `lab_utils/routing_tool.py` | Gợi ý semantic routing (tư vấn) |
| Auto `trace_id` | `adk_callbacks.py` | Tự sinh trace cho ADK Web (governance A2A) |

Chạy cell **kiểm tra trước khi mở ADK Web** bên dưới, rồi làm bài tập ghi kết quả.

In [ ]:
# Kiểm tra trước khi mở ADK Web — chạy cell này trong notebook
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import httpx
from lab_utils.routing_tool import suggest_routing

CAPSTONE_CARDS = {
    "search_agent": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8003/.well-known/agent-card.json",
}

print("=== Kiểm tra A2A specialists ===")
all_ok = True
for name, url in CAPSTONE_CARDS.items():
    try:
        r = httpx.get(url, timeout=5)
        r.raise_for_status()
        print(f"✓ {name}")
    except Exception as exc:
        all_ok = False
        print(f"✗ {name}: {exc}")

print("\n=== Demo suggest_routing (orchestrator tool) ===")
for q in [
    "Tìm bài viết về MCP",
    "SELECT avg latency FROM agent_metrics",
    "Viết báo cáo tổng hợp kết quả nghiên cứu",
]:
    result = suggest_routing(q)
    print(f"  '{q[:40]}...' → {result['recommended_agent']} (top score: {result['top_candidates'][0]['score']})")

if all_ok:
    print("\n✓ Sẵn sàng — mở terminal và chạy: bash scripts/start_capstone.sh")
    print("  Sau đó truy cập http://localhost:8000")
else:
    print("\n✗ Chưa sẵn sàng — chạy: bash scripts/start_a2a_servers.sh")

### 📝 Bài tập sinh viên — Ghi kết quả ADK Web

**Hướng dẫn:** Sau khi `start_capstone.sh` chạy, mở http://localhost:8000. **Tạo session mới** (nút +), gõ từng prompt bên dưới.

**Đọc kết quả trong ADK Web:**
| Bubble | Ý nghĩa |
|--------|---------|
| `State: trace_id, task_id...` | Governance khởi tạo session — **không phải câu trả lời** |
| Text từ orchestrator | Câu trả lời chính |
| Tab **Trace** (bên phải) | Xem `transfer_to_agent`, MCP calls, A2A events |

Nếu chat trống sau prompt → mở **Trace**, kiểm tra có `transfer_to_agent` không. Restart: `Ctrl+C` rồi `bash scripts/start_capstone.sh`.

| # | Prompt (copy vào ADK Web) | Kỳ vọng |
|---|---------------------------|---------|
| **W1** | `Tôi cần tìm web về multi-agent orchestration. Hãy transfer_to_agent sang search_agent và trả kết quả.` | A2A → search_agent + text |
| **W2** | `Bước 1: dùng search_documents tìm MCP. Bước 2: dùng sql_query SELECT * FROM agent_metrics. Bước 3: tóm tắt báo cáo ngắn.` | MCP: search_documents + sql_query |
| **W3** | `Ủy quyền synthesis_agent tổng hợp báo cáo executive từ các findings về MCP và A2A.` | A2A → synthesis_agent |
| **W4** | `Gọi suggest_routing rồi giải thích bạn sẽ chọn agent nào: "SELECT độ trễ trung bình từ agent_metrics"` | Tool suggest_routing → database_agent |
| **W5** | `DROP TABLE agent_metrics` (thử governance) | Bị chặn — blocked / deny |

**Quan sát thêm (ghi vào bảng):**
- Trace ID có trong session state không?
- Agent nào xuất hiện trong Trace? (`orchestrator`, `search_agent`, …)
- `tail -5 logs/governance_audit.jsonl`

In [1]:
# ADK Web results - filled from the verified local run

adk_web_results = [
    {
        "prompt_id": "W1",
        "agents_involved": ["orchestrator", "search_agent"],
        "tools_or_protocol": "A2A transfer_to_agent",
        "outcome": "DAT",
        "notes": "ADK Web shows transfer_to_agent and orchestrator -> search_agent. Screenshot: artifacts/screenshots/adk_web_w1_chat.png",
    },
    {
        "prompt_id": "W2",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "MCP (search_documents, sql_query, summarize_text)",
        "outcome": "DAT",
        "notes": "ADK Web shows search_documents, sql_query, summarize_text; final answer includes 820/1100/2400 ms metrics. Screenshot: artifacts/screenshots/adk_web_w2_chat.png",
    },
    {
        "prompt_id": "W3",
        "agents_involved": ["orchestrator", "synthesis_agent"],
        "tools_or_protocol": "A2A -> synthesis_agent",
        "outcome": "DAT",
        "notes": "Backend trace c3f37856-08c1-46aa-a8c5-e4bc4b84a243 returned executive report from synthesis_agent.",
    },
    {
        "prompt_id": "W4",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "suggest_routing",
        "outcome": "DAT",
        "notes": "Raw trace shows function_call suggest_routing; recommended_agent = database_agent, score = 0.531.",
    },
    {
        "prompt_id": "W5",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "SQL governance deny",
        "outcome": "DAT",
        "notes": "DROP TABLE is refused by orchestrator/read-only SQL guard; direct MCP call returns governance=deny.",
    },
]

screenshot_paths = [
    "artifacts/screenshots/adk_web_w1_chat.png",
    "artifacts/screenshots/adk_web_w1_trace.png",
    "artifacts/screenshots/adk_web_w2_chat.png",
    "artifacts/screenshots/adk_web_w2_trace.png",
]

print(f"{'ID':<4} {'Agents':<35} {'Protocol':<42} {'Result':<8} Notes")
print("-" * 130)
for row in adk_web_results:
    agents = ", ".join(row["agents_involved"])
    print(f"{row['prompt_id']:<4} {agents:<35} {row['tools_or_protocol']:<42} {row['outcome']:<8} {row['notes']}")

passed = sum(1 for r in adk_web_results if r["outcome"] == "DAT")
print(f"\nTotal: {passed}/{len(adk_web_results)} prompts passed")
print("\nScreenshots:")
for path in screenshot_paths:
    print("-", path)


ID   Agents                              Protocol                                   Result   Notes
----------------------------------------------------------------------------------------------------------------------------------
W1   orchestrator, search_agent          A2A transfer_to_agent                      DAT      ADK Web shows transfer_to_agent and orchestrator -> search_agent. Screenshot: artifacts/screenshots/adk_web_w1_chat.png
W2   orchestrator                        MCP (search_documents, sql_query, summarize_text) DAT      ADK Web shows search_documents, sql_query, summarize_text; final answer includes 820/1100/2400 ms metrics. Screenshot: artifacts/screenshots/adk_web_w2_chat.png
W3   orchestrator, synthesis_agent       A2A -> synthesis_agent                     DAT      Backend trace c3f37856-08c1-46aa-a8c5-e4bc4b84a243 returned executive report from synthesis_agent.
W4   orchestrator                        suggest_routing                            DAT      Raw trace s

---
## Module 5 — State, Bảo mật & Observability

### Quản lý State

| Mẫu | Trường hợp dùng |
|-----|----------------|
| **Agent stateless** | Scale ngang, K8s auto-scale |
| **State ngoài (Redis)** | Ngữ cảnh ngắn hạn (<1 giờ) |
| **PostgreSQL** | Lịch sử hội thoại lâu dài |
| **Session ID** | Sticky routing khi stateful |

**Thực hành tốt:** Thiết kế stateless trước; externalize state khi cần.

Trong ADK, dùng `tool_context.state` trong function tool để chia sẻ dữ liệu giữa các lượt.

In [ ]:
# Ví dụ: state qua ToolContext (mẫu khái niệm)
example_tool = '''
from google.adk.tools.tool_context import ToolContext

async def track_cost(amount: float, tool_context: ToolContext) -> str:
    """Cộng dồn chi phí API vào session state."""
    total = tool_context.state.get("total_cost", 0.0) + amount
    tool_context.state["total_cost"] = total
    if total > 10.0:
        return f"Đã đạt trần chi phí (${total:.2f}). Cần phê duyệt của người."
    return f"Đã cập nhật chi phí: ${total:.2f}"
'''
print(example_tool)


from google.adk.tools.tool_context import ToolContext

async def track_cost(amount: float, tool_context: ToolContext) -> str:
    """Cộng dồn chi phí API vào session state."""
    total = tool_context.state.get("total_cost", 0.0) + amount
    tool_context.state["total_cost"] = total
    if total > 10.0:
        return f"Đã đạt trần chi phí (${total:.2f}). Cần phê duyệt của người."
    return f"Đã cập nhật chi phí: ${total:.2f}"



### Bảo mật & Governance

| Kiểm soát | Triển khai |
|-----------|------------|
| **Rate limiting** | Giới hạn gọi tool theo agent & user |
| **Sandbox execution** | Code agent trong Docker, không network |
| **Human-in-the-loop** | Hành động quan trọng cần phê duyệt |
| **Audit log** | Timestamp, agent ID, I/O cho mọi lần gọi tool |
| **Minimal capability** | Agent chỉ có tool cần thiết |

**Chống chạy vô hạn:** tối đa 50 lần gọi tool/task, tối đa 5 phút thực thi, trần chi phí mỗi request.

### Ma trận Capability Agent (từ slide)

| Agent | Được phép | Bị chặn |
|-------|-----------|--------|
| Research | search, đọc | ghi |
| Database | chỉ SELECT | DDL, DML |
| Email | soạn thảo | gửi (cần phê duyệt) |
| Code | chạy trong sandbox | deploy production |

### Data Governance — MCP & A2A (thực hành)

Hệ thống governance nằm trong `lab_utils/governance/`:

| Thành phần | Vai trò |
|------------|--------|
| `policy.json` | Ma trận capability: ai được gọi MCP/A2A tool nào |
| `GovernanceGuard` | Kiểm tra trước mỗi lần gọi (SQL read-only, rate limit, PII) |
| `AuditLogger` | Ghi audit vào `logs/governance_audit.jsonl` |
| `adk_callbacks.py` | `before_tool_callback` chặn tool vi phạm policy |

**Luồng kiểm soát:**

```
Request → GovernanceGuard → [ALLOW | DENY | HITL_REQUIRED]
                ↓
         AuditLogger (timestamp, actor, I/O)
                ↓
         MCP tool / A2A dispatch
```

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.governance import get_guard, AuditLogger

guard = get_guard()
guard.rate_limiter.reset()
audit = AuditLogger()

print("=== MCP matrix (orchestrator) ===")
conn = guard.authorize_mcp_connection("orchestrator")
print(f"MCP connection: {conn.verdict.value} - tools: {conn.metadata.get('allowed_tools')}")

print("
=== SQL violation check (MCP) ===")
bad_sql = guard.authorize_mcp_tool(
    "orchestrator", "sql_query", {"sql": "DROP TABLE agent_metrics"}
)
print(f"DROP TABLE -> {bad_sql.verdict.value}: {bad_sql.reason}")

good_sql = guard.authorize_mcp_tool(
    "orchestrator", "sql_query", {"sql": "SELECT * FROM agent_metrics"}
)
print(f"Valid SELECT -> {good_sql.verdict.value}")

print("
=== A2A dispatch check ===")
allowed = guard.authorize_a2a_dispatch(
    "orchestrator", "search_agent", trace_id="demo-trace-001"
)
print(f"orchestrator -> search_agent: {allowed.verdict.value}")

blocked = guard.authorize_a2a_dispatch(
    "orchestrator", "email_agent", trace_id="demo-trace-001"
)
print(f"orchestrator -> email_agent: {blocked.verdict.value}: {blocked.reason}")

no_trace = guard.authorize_a2a_dispatch("orchestrator", "database_agent")
print(f"Missing trace_id -> {no_trace.verdict.value}: {no_trace.reason}")

print("
=== PII -> HITL ===")
pii_sql = guard.authorize_mcp_tool(
    "orchestrator",
    "sql_query",
    {"sql": "SELECT * FROM agent_metrics WHERE email = 'user@vinuni.edu.vn'"},
)
print(f"PII in SQL -> {pii_sql.verdict.value}: {pii_sql.reason}")

print("
=== Audit summary ===")
print(audit.summary())


### 📝 Bài tập 5.2 — Mở rộng chính sách governance

1. Mở `lab_utils/governance/policy.json` và thêm agent `synthesis_agent` vào `allowed_targets` của orchestrator.
2. Thêm rule chặn từ khóa `password` trong `search_documents`.
3. Chạy lại cell trên và xác nhận audit log ghi đủ sự kiện `deny` / `hitl_required`.
4. *(Nâng cao)* Viết test đảm bảo caller không hợp lệ không mở được kết nối MCP.

In [ ]:
# Kiem tra Bai tap 5.2 - mo rong chinh sach governance
guard.rate_limiter.reset()

print("=== 1) synthesis_agent in allowed_targets ===")
d = guard.authorize_a2a_dispatch("orchestrator", "synthesis_agent", trace_id="t-1")
print("orchestrator -> synthesis_agent:", d.verdict.value)
assert d.verdict.value == "allow"

print("
=== 2) Block keyword 'password' in search_documents ===")
blocked = guard.authorize_mcp_tool("orchestrator", "search_documents", {"query": "tim password cua admin"})
print("query with password ->", blocked.verdict.value, "-", blocked.reason)
assert blocked.verdict.value == "deny"

ok = guard.authorize_mcp_tool("orchestrator", "search_documents", {"query": "MCP transport"})
print("normal query ->", ok.verdict.value)
assert ok.verdict.value == "allow"

print("
=== 3) Audit log contains deny/hitl_required ===")
print(audit.summary())

print("
=== 4) Advanced: invalid caller cannot open MCP ===")
rogue = guard.authorize_mcp_connection("rogue_actor")
print("rogue_actor opens MCP ->", rogue.verdict.value, "-", rogue.reason)
assert rogue.verdict.value == "deny", "invalid caller must be denied"
print("Test PASS: invalid caller is denied when opening MCP")


### Observability — Distributed Tracing

Một **trace ID** duy nhất nên bao trùm toàn bộ request multi-agent:

```
Trace ID: abc-123
└── Orchestrator (2.3s)
    ├── Search Agent (0.8s)
    │   └── web_search (0.5s)
    └── DB Agent (1.1s)
        └── sql_query (0.9s)
```

ADK truyền metadata qua ranh giới A2A qua `RunConfig.custom_metadata`.

In [ ]:
import uuid

from google.adk.agents.run_config import RunConfig

trace_id = str(uuid.uuid4())
run_config = RunConfig(
    custom_metadata={
        "trace_id": trace_id,
        "lab": "day26",
        "user_tier": "student",
    }
)

print(f"Trace ID: {trace_id}")
print(f"Các khóa metadata: {list(run_config.custom_metadata.keys())}")
print("\nTrên A2A agent remote, metadata xuất hiện dạng:")
print('  event.custom_metadata["a2a_metadata"]["trace_id"]')

### Các metric cần theo dõi

| Metric | Mục đích |
|--------|----------|
| `tasks_completed` / `tasks_failed` | Độ tin cậy |
| `avg_task_duration` | Độ trễ |
| `tool_call_count` | Phát hiện chạy vô hạn |
| `cost_per_task` | Phân bổ ngân sách |
| `queue_depth` | Lập kế hoạch năng lực |

### 📝 Bài tập 5.1 — Thiết kế chính sách Governance

Với hệ 4-agent nghiên cứu, viết ma trận capability:
1. Tool nào mỗi agent được gọi
2. Hành động nào cần phê duyệt người
3. Rate limit theo agent
4. Số lần gọi tool và thời gian thực thi tối đa

#### ✅ Trả lời Bài tập 5.1

**1) Tool mỗi agent được gọi**

| Agent | MCP tools (research-tools, stdio) | A2A tools nội bộ | A2A dispatch được phép |
|---|---|---|---|
| `orchestrator` | `search_documents`, `sql_query` (chỉ SELECT bảng `agent_metrics`), `summarize_text`, `count_words` | — | `search_agent`, `database_agent`, `synthesis_agent` |
| `search_agent` | — (không mở kết nối MCP) | `search_web` | — |
| `database_agent` | — | `run_sql_query` (chỉ SELECT bảng `agent_metrics`) | — |
| `synthesis_agent` | — | `synthesize_report` | — |

**2) Hành động cần phê duyệt người (HITL) / bị chặn hẳn (DENY)**

| Tình huống | Verdict | Lý do |
|---|---|---|
| SQL không phải SELECT (INSERT/UPDATE/DELETE/DROP/ALTER/CREATE/TRUNCATE) | DENY | Chỉ read-only trên `agent_metrics` |
| SQL chứa PII (email, SSN-like pattern) | HITL_REQUIRED | Rò rỉ dữ liệu nhạy cảm qua query |
| `search_documents` chứa từ khóa `password` | DENY | Chặn truy vấn tìm kiếm thông tin nhạy cảm (Bài tập 5.2) |
| A2A dispatch thiếu `trace_id` | HITL_REQUIRED | Bắt buộc traceability trước khi cho phép |
| A2A dispatch tới agent ngoài `allowed_targets`/`allowed_callers` | DENY | Vi phạm capability matrix |
| Chi phí task vượt `cost_ceiling_usd` (10.0) | HITL_REQUIRED | Chống vượt ngân sách |
| Vượt `max_tool_calls_per_task` (50) | DENY | Chống chạy vô hạn (runaway) |

**3) Rate limit theo agent**

Đồng nhất `rate_limit_per_minute = 30` cho mọi actor (orchestrator + 3 specialist), áp dụng cho cả MCP tool call lẫn A2A dispatch (`GovernanceGuard._check_rate_limit`, dùng chung `RateLimiter`).

**4) Giới hạn số lần gọi tool & thời gian thực thi tối đa**

- `max_tool_calls_per_task = 50` — tổng MCP tool call + A2A dispatch trong một `task_id` (chống loop vô hạn giữa các agent).
- `max_execution_seconds = 300` (5 phút) — trần thời gian thực thi một task đầu-cuối.
- `cost_ceiling_usd = 10.0` — trần chi phí tích lũy mỗi task, vượt trần → HITL thay vì cắt cứng, cho phép người duyệt tiếp tục nếu hợp lý.

> Toàn bộ 4 mục trên khớp với `lab_utils/governance/policy.json` (`global_limits` + `connections.mcp` + `connections.a2a`) hiện có trong repo — đây là chính sách đang thực sự được `GovernanceGuard` enforce, không phải đề xuất lý thuyết.


---
## Lab #26 Capstone - Multi-Agent Infrastructure

**Goal:** Build a 4-agent system (orchestrator + 3 specialists) through A2A + MCP with full trace and ADK Web demo.

**Duration:** 2 hours

> Before the check cell below, A2A servers must be running (Module 0.5). Recommended capstone:
> ```bash
> bash scripts/start_capstone.sh
> ```
> or `bash scripts/start_a2a_servers.sh` then `bash scripts/start_adk_web.sh`.

### Checklist

- [x] MCP server with 4 tools (stdio spawned automatically by McpToolset)
- [x] Agent registry with health check
- [x] Semantic router + `suggest_routing` tool on orchestrator
- [x] Search agent exposed through `to_a2a()` on port 8001
- [x] Database agent exposed through `to_a2a()` on port 8002
- [x] **Synthesis agent** exposed through `to_a2a()` on port 8003
- [x] Orchestrator consumes all three through `RemoteA2aAgent`
- [x] ADK Web demo - 5 prompts W1-W5 recorded in Module 4
- [x] Trace ID auto-created in ADK Web (governance auto-init)
- [x] Governance policy writes audit (`logs/governance_audit.jsonl`)

### Stretch challenges

1. **Transport SSE:** Implement MCP server with FastAPI + uvicorn on port 8080
2. **Concurrent load:** Send 5 research requests and observe routing distribution
3. **Embedding router:** Replace bag-of-words with `text-embedding-004`
4. **HITL gate:** Pause before actions above the $10 API cost ceiling


In [ ]:
# Script kiểm tra capstone — chạy khi A2A servers đã lên (Module 0.5)
import httpx

checks = {
    "search_agent_card": "http://localhost:8001/.well-known/agent-card.json",
    "database_agent_card": "http://localhost:8002/.well-known/agent-card.json",
    "synthesis_agent_card": "http://localhost:8003/.well-known/agent-card.json",
}

START_HINT = """
✗ A2A servers chưa chạy. Làm một trong các cách sau:

  Cách 1 (khuyến nghị capstone):
    bash scripts/start_capstone.sh

  Cách 2 (notebook): quay lại cell「Khởi động A2A servers nền」ở Module 0.5

  Cách 3 (terminal tại thư mục dự án):
    bash scripts/start_a2a_servers.sh

  Cách 4 (ba terminal):
    uvicorn agents.search_agent.agent:a2a_app --host localhost --port 8001
    uvicorn agents.database_agent.agent:a2a_app --host localhost --port 8002
    uvicorn agents.synthesis_agent.agent:a2a_app --host localhost --port 8003
"""

all_ok = True
for name, url in checks.items():
    try:
        r = httpx.get(url, timeout=5)
        ok = r.status_code == 200
        err = ""
    except Exception as exc:
        ok = False
        err = str(exc)
    status = "ĐẠT" if ok else "CHƯA ĐẠT"
    print(f"[{status}] {name}" + (f" — {err}" if err else ""))
    all_ok = all_ok and ok

if all_ok:
    print("\n✓ Sẵn sàng demo capstone")
    print("  Tiếp theo: bash scripts/start_capstone.sh → http://localhost:8000")
else:
    print(START_HINT)

---
## Tổng kết — Điểm then chốt

1. **MCP** chuẩn hóa giao diện tool — build một lần, dùng trên nhiều LLM framework. Triển khai MCP server trên K8s với mẫu registry.
2. **A2A** là microservices cho AI — hợp đồng, auth, observability. Google ADK expose/tiêu thụ agent qua `to_a2a()` và `RemoteA2aAgent`.
3. **Trí tuệ routing** ở lớp orchestrator tốt hơn chọn agent cứng — đầu tư semantic routing + fallback chain.

---

## Xem trước buổi sau

**Ngày 27:** Data Observability & Advanced Monitoring — Great Expectations, phát hiện bất thường, dbt tests, SLO engineering.

### Bài tập về nhà

- Hoàn thành checklist capstone Lab #26
- Đọc: [Tài liệu Great Expectations checkpoint](https://docs.greatexpectations.io/)
- Đọc: Google SRE Workbook — chương SLO engineering